In [ ]:
import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)
import matplotlib.pyplot as plt
import time

data=pd.read_csv('/opt/neurothink/neuro-deb05933716f4eb6be138022abf35d2f/train.csv')
test=pd.read_csv('/opt/neurothink/neuro-1d4c617f8ef0447bafed400cb9e1ff6b/test.csv')

data['Age'].fillna(data['Age'].median(),inplace=True)
data.drop('Cabin',axis=1,inplace=True)
data['Embarked'].fillna('S',inplace=True)
data.drop(['PassengerId','Name','Ticket'],axis=1,inplace=True)
data['Pclass']=data['Pclass'].astype(str)
data['Accompanied']=0
for i in range(data.shape[0]):
    if (data.loc[i,'SibSp']+data.loc[i,'Parch'])>0:
        data.loc[i,'Accompanied']=1
data['Accompanied']=data['Accompanied'].astype(str)
data.drop(['SibSp','Parch'],axis=1,inplace=True)
data=pd.get_dummies(data)

X=data[['Age','Fare','Pclass_1','Pclass_2','Pclass_3','Sex_female','Sex_male','Embarked_C','Embarked_Q','Embarked_S','Accompanied_0','Accompanied_1']]
y=data['Survived']

from sklearn.model_selection import GridSearchCV
from xgboost.sklearn import XGBClassifier
import lightgbm as lgb
from sklearn.ensemble import GradientBoostingClassifier as gbr

xgb=XGBClassifier(learning_rate=0.1,n_estimators=500,max_depth=5,min_child_weight=1,gamma=0,
subsample=0.8,colsample_bytree=0.8,objective= 'binary:logistic',seed=0)

start = time.time_ns()
param_test = {'n_estimators':range(30,40,50)}
gs = GridSearchCV(estimator = xgb,param_grid = param_test, scoring='roc_auc',n_jobs=-1,cv=5)
gs.fit(X,y)
print(gs.best_params_,gs.best_score_)
print(gs.cv_results_['mean_test_score'])
end = time.time_ns()
print('**************')
print('TIME USED (ms):', (end-start)/1000)



In [ ]:
#!pip install catboost
from catboost import CatBoostClassifier

cat = CatBoostClassifier(eval_metric='Accuracy', use_best_model=True,  random_seed=42)

param_test = {'iterations':[30,40,50], 'use_best_model': [False]}
"""
params = {'depth':[3,1,2,6,4,5,7,8,9,10],
          'iterations':[250,100,500,1000],
          'learning_rate':[0.03,0.001,0.01,0.1,0.2,0.3], 
          'l2_leaf_reg':[3,1,5,10,100],
          'border_count':[32,5,10,20,50,100,200],
          'ctr_border_count':[50,5,10,20,100,200],
          'thread_count':4}
"""
start = time.time_ns()
gs_cat = GridSearchCV(estimator = cat, param_grid = param_test, scoring='roc_auc',n_jobs=-1,cv=5)
gs_cat.fit(X,y)
print(gs_cat.best_params_,gs_cat.best_score_)
print(gs_cat.cv_results_['mean_test_score'])
end = time.time_ns()
print('**************')
print('TIME USED (ms):', (end-start)/1000)

In [ ]:
orig_data=pd.read_csv('/opt/neurothink/neuro-deb05933716f4eb6be138022abf35d2f/train.csv')
orig_data.head()
for col in orig_data:
    orig_data[col] = orig_data[col].fillna('NaN')
train_X = orig_data.loc[:, orig_data.columns!='Survived']
train_Y = orig_data['Survived']
train_X.head()
categorical_indices = [0,1,2,3,5,6,7,9,10]
start = time.time_ns()
gs_cat = GridSearchCV(estimator = cat, param_grid = param_test, scoring='roc_auc',n_jobs=-1,cv=5)
gs_cat.fit(train_X,train_Y, cat_features=categorical_indices)
end = time.time_ns()
print(gs_cat.best_params_,gs_cat.best_score_)
print(gs_cat.cv_results_['mean_test_score'])
print('**************')
print('TIME USED (ms):', (end-start)/1000)


In [ ]:
from sklearn.ensemble import RandomForestClassifier
rf = RandomForestClassifier(max_depth=2, random_state=0)
start = time.time_ns()
param_test = {"max_depth":[2,3,4]}
gs_rf = GridSearchCV(estimator = rf, param_grid = param_test, scoring='roc_auc',n_jobs=-1,cv=5)
gs_rf.fit(X,y)
end = time.time_ns()
print(gs_rf.best_params_,gs_rf.best_score_)
print(gs_rf.cv_results_['mean_test_score'])
print('**************')
print('TIME USED (ms):', (end-start)/1000)